In [ ]:
!git clone https://github.com/ultralytics/yolov5  # clone repo
%cd yolov5
!pip install -r requirements.txt  # install dependencies

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import matplotlib.image as mpimg
import os

In [ ]:
# Reading the contents of coco128.yaml
yaml_path = '/kaggle/input/rider-with-helmet-without-helmet-number-plate/coco128.yaml'

with open(yaml_path, 'r') as file:
    yaml_contents = file.read()
    
print(yaml_contents)

In [ ]:
# Function to count class instances in a given folder
def count_class_instances(folder_path):
    labels_folder = Path(folder_path) / 'labels'
    class_counts = Counter()

    for label_file in labels_folder.glob('*.txt'):
        with open(label_file, 'r') as file:
            annotations = file.readlines()
            for annotation in annotations:
                class_id = int(annotation.split()[0])
                class_counts[class_id] += 1

    return class_counts

# Example usage
train_class_counts = count_class_instances('/kaggle/input/rider-with-helmet-without-helmet-number-plate/train')
val_class_counts = count_class_instances('/kaggle/input/rider-with-helmet-without-helmet-number-plate/val')

In [ ]:
print("Train Counts: ",train_class_counts)
print("Validation Counts: ",val_class_counts)

- Class 2: 120 instances
- Class 3: 116 instances
- Class 1: 93 instances
- Class 0: 64 instances

In [ ]:
# Function for augmenting images (horizontal flip and brightness adjustment)
def augment_image(image):
    flipped = cv2.flip(image, 1)

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    v = cv2.add(v, np.random.randint(30, 70))
    final_hsv = cv2.merge((h, s, v))
    bright_adjusted = cv2.cvtColor(final_hsv, cv2.COLOR_HSV2BGR)

    return flipped, bright_adjusted

# Function to demonstrate augmentation on a sample image
def demonstrate_augmentation(folder_path, class_id=0):
    labels_folder = Path(folder_path) / 'labels'
    images_folder = Path(folder_path) / 'images'

    for label_file in labels_folder.glob('*.txt'):
        with open(label_file, 'r') as file:
            annotations = file.readlines()
            if any(int(ann.split()[0]) == class_id for ann in annotations):
                image_name = label_file.stem
                image_path = images_folder / f'{image_name}.jpg'
                img = cv2.imread(str(image_path))
                flipped, bright_adjusted = augment_image(img)
                return img, flipped, bright_adjusted

    return None, None, None

In [ ]:
# Displaying the original and augmented images
original, flipped, bright_adjusted = demonstrate_augmentation('/kaggle/input/rider-with-helmet-without-helmet-number-plate/train')
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(flipped, cv2.COLOR_BGR2RGB))
axes[1].set_title('Flipped')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(bright_adjusted, cv2.COLOR_BGR2RGB))
axes[2].set_title('Brightness Adjusted')
axes[2].axis('off')

plt.show()

In [ ]:
# Navigate to the YOLOv5 directory
%cd /kaggle/working/yolov5

# Create a YAML file for your dataset
data_yaml_content = """
train: /kaggle/input/rider-with-helmet-without-helmet-number-plate/train/images
val: /kaggle/input/rider-with-helmet-without-helmet-number-plate/val/images

nc: 4
names: ['with helmet', 'without helmet', 'rider', 'number plate']
"""

# Write the dataset configuration to a YAML file
with open('data.yaml', 'w') as file:
    file.write(data_yaml_content)

# Run name and project paths (used by all downstream cells)
RUN_NAME    = 'yolov5_rider_safety'
PROJECT_DIR = '/kaggle/working/yolov5/runs/train'
RUN_DIR     = f'{PROJECT_DIR}/{RUN_NAME}'

# Command to train the YOLOv5 model.
# Flags added for full metric logging:
#   --save-period 1 : save a checkpoint every epoch
#   --exist-ok      : overwrite an old run with the same name (clean re-runs)
#   --patience 0    : disable early stopping so all 50 epochs are logged
# YOLOv5 already prints a per-epoch table (box/obj/cls loss, P, R, mAP@0.5, mAP@0.5:0.95)
# and writes them to results.csv inside the run directory.
!python train.py --img 640 --batch 16 --epochs 50 \
                 --data data.yaml --cfg yolov5s.yaml \
                 --weights yolov5s.pt \
                 --project {PROJECT_DIR} --name {RUN_NAME} --exist-ok \
                 --save-period 1 --patience 0


In [ ]:
import pandas as pd
from pathlib import Path

# Path to the YOLOv5 run directory (set in the training cell above)
RUN_DIR    = '/kaggle/working/yolov5/runs/train/yolov5_rider_safety'
RESULTS_CSV = Path(RUN_DIR) / 'results.csv'

# A folder to drop all report-ready figures into
REPORT_DIR = Path(RUN_DIR) / 'report_figures'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load and clean the per-epoch metrics
results = pd.read_csv(RESULTS_CSV)
results.columns = [c.strip() for c in results.columns]   # YOLOv5 pads column names with spaces
print(f"Loaded {len(results)} epochs of metrics from {RESULTS_CSV}")
print("Columns available:", list(results.columns))

# Show the full per-epoch metric table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
results.round(4)


In [ ]:
# Final-epoch and best-epoch metric summary
metric_cols = [c for c in results.columns if c.startswith('metrics/')]
loss_cols   = [c for c in results.columns if 'loss' in c]

final_row = results.iloc[-1]
best_map_idx = results['metrics/mAP_0.5'].idxmax() if 'metrics/mAP_0.5' in results else None

print("=" * 60)
print("FINAL EPOCH METRICS")
print("=" * 60)
for col in metric_cols:
    print(f"  {col:<28s}: {final_row[col]:.4f}")
for col in loss_cols:
    print(f"  {col:<28s}: {final_row[col]:.4f}")

if best_map_idx is not None:
    print()
    print("=" * 60)
    print(f"BEST EPOCH (by mAP@0.5)  -- epoch index {best_map_idx}")
    print("=" * 60)
    best_row = results.iloc[best_map_idx]
    for col in metric_cols:
        print(f"  {col:<28s}: {best_row[col]:.4f}")


In [ ]:
import matplotlib.pyplot as plt

# Common plot settings for report-quality figures
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

epochs = results['epoch'] if 'epoch' in results.columns else results.index

def plot_metric(y_cols, title, ylabel, fname, colors=None):
    """Plot one or more columns from `results` against epoch and save the figure."""
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, col in enumerate(y_cols):
        if col not in results.columns:
            continue
        c = colors[i] if colors and i < len(colors) else None
        ax.plot(epochs, results[col], label=col.split('/')[-1], linewidth=2, color=c)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    out = REPORT_DIR / fname
    fig.savefig(out)
    plt.show()
    print(f"saved -> {out}")

# 1. Training losses
plot_metric(
    ['train/box_loss', 'train/obj_loss', 'train/cls_loss'],
    'Training Losses vs Epoch', 'Loss', 'fig_train_losses.png',
    colors=['tab:blue', 'tab:orange', 'tab:green'],
)


In [ ]:
# 2. Validation losses
plot_metric(
    ['val/box_loss', 'val/obj_loss', 'val/cls_loss'],
    'Validation Losses vs Epoch', 'Loss', 'fig_val_losses.png',
    colors=['tab:blue', 'tab:orange', 'tab:green'],
)


In [ ]:
# 3. Train vs Validation loss comparison (one subplot per loss component)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
loss_pairs = [
    ('train/box_loss', 'val/box_loss', 'Box Loss'),
    ('train/obj_loss', 'val/obj_loss', 'Objectness Loss'),
    ('train/cls_loss', 'val/cls_loss', 'Classification Loss'),
]
for ax, (tr, va, name) in zip(axes, loss_pairs):
    if tr in results.columns:
        ax.plot(epochs, results[tr], label='train', linewidth=2, color='tab:blue')
    if va in results.columns:
        ax.plot(epochs, results[va], label='val',   linewidth=2, color='tab:red')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(name)
    ax.legend()
fig.suptitle('Train vs Validation Losses', fontsize=13, y=1.02)
fig.tight_layout()
out = REPORT_DIR / 'fig_train_vs_val_loss.png'
fig.savefig(out)
plt.show()
print(f"saved -> {out}")


In [ ]:
# 4. Precision and Recall
plot_metric(
    ['metrics/precision', 'metrics/recall'],
    'Precision and Recall vs Epoch', 'Score', 'fig_precision_recall.png',
    colors=['tab:purple', 'tab:cyan'],
)


In [ ]:
# 5. mAP@0.5 and mAP@0.5:0.95
plot_metric(
    ['metrics/mAP_0.5', 'metrics/mAP_0.5:0.95'],
    'Mean Average Precision vs Epoch', 'mAP', 'fig_map.png',
    colors=['tab:green', 'tab:red'],
)


In [ ]:
# 6. One figure per individual detection metric (handy for slides / report appendix)
individual_metrics = [
    ('metrics/precision',     'Precision',     'fig_metric_precision.png',     'tab:purple'),
    ('metrics/recall',        'Recall',        'fig_metric_recall.png',        'tab:cyan'),
    ('metrics/mAP_0.5',       'mAP @ 0.5',     'fig_metric_map50.png',         'tab:green'),
    ('metrics/mAP_0.5:0.95',  'mAP @ 0.5:0.95','fig_metric_map5095.png',       'tab:red'),
]
for col, title, fname, color in individual_metrics:
    if col not in results.columns:
        continue
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(epochs, results[col], linewidth=2.2, color=color)
    best_idx = results[col].idxmax()
    ax.scatter([epochs.iloc[best_idx] if hasattr(epochs, 'iloc') else best_idx],
               [results[col].iloc[best_idx]], color=color, s=70, zorder=5,
               label=f'best = {results[col].iloc[best_idx]:.4f}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.set_title(f'{title} vs Epoch')
    ax.legend()
    fig.tight_layout()
    out = REPORT_DIR / fname
    fig.savefig(out)
    plt.show()
    print(f"saved -> {out}")


In [ ]:
# 7. Learning rate schedule
lr_cols = [c for c in results.columns if c.startswith('x/lr')]
if lr_cols:
    plot_metric(lr_cols, 'Learning Rate Schedule', 'Learning rate', 'fig_learning_rate.png')
else:
    print("No learning-rate columns found in results.csv")


In [ ]:
# Display YOLOv5's built-in evaluation figures.
# We use IPython.display.Image (which embeds raw bytes) instead of mpimg.imread,
# because some Pillow versions throw AttributeError: _im on YOLOv5's JPEGs.
import os
from IPython.display import Image, Markdown, display

builtin_plots = [
    ('results.png',          'Combined training curves (YOLOv5 default)'),
    ('confusion_matrix.png', 'Confusion matrix on the validation set'),
    ('PR_curve.png',         'Precision-Recall curve'),
    ('F1_curve.png',         'F1 vs confidence threshold'),
    ('P_curve.png',          'Precision vs confidence threshold'),
    ('R_curve.png',          'Recall vs confidence threshold'),
    ('labels.jpg',           'Label distribution and bbox statistics'),
    ('labels_correlogram.jpg', 'Label correlogram'),
    ('val_batch0_labels.jpg',  'Validation batch 0 - ground truth'),
    ('val_batch0_pred.jpg',    'Validation batch 0 - predictions'),
]

for fname, caption in builtin_plots:
    fpath = os.path.join(RUN_DIR, fname)
    if not os.path.exists(fpath):
        print(f"[skip] {fname} not found")
        continue
    try:
        display(Markdown(f"**{caption}**  \n`{fpath}`"))
        display(Image(filename=fpath, width=900))
    except Exception as e:
        # Last-resort fallback: try cv2 + matplotlib so we still see *something*
        print(f"[warn] could not embed {fname} directly ({e}); trying cv2 fallback")
        try:
            import cv2, matplotlib.pyplot as plt
            img = cv2.imread(fpath)
            if img is None:
                print(f"[warn] cv2 also failed to read {fname}")
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            fig, ax = plt.subplots(figsize=(10, 7))
            ax.imshow(img); ax.set_title(caption); ax.axis('off')
            plt.show()
        except Exception as e2:
            print(f"[warn] fallback also failed for {fname}: {e2}")


In [ ]:
# Run validation on the best checkpoint and save plots/metrics under runs/val/
BEST_WEIGHTS = f'{RUN_DIR}/weights/best.pt'

!python val.py --weights {BEST_WEIGHTS} \
               --data data.yaml --img 640 --task val \
               --project /kaggle/working/yolov5/runs/val \
               --name yolov5_rider_safety_eval --exist-ok \
               --save-txt --save-conf --verbose


In [ ]:
import shutil
from pathlib import Path

REPORT_DIR = Path(RUN_DIR) / 'report_figures'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# (yolov5 source filename, report filename used in results.tex)
copy_map = [
    ('results.png',            'fig_results_combined.png'),
    ('confusion_matrix.png',   'fig_confusion_matrix.png'),
    ('PR_curve.png',           'fig_pr_curve.png'),
    ('F1_curve.png',           'fig_f1_curve.png'),
    ('P_curve.png',            'fig_p_curve.png'),
    ('R_curve.png',            'fig_r_curve.png'),
    ('labels.jpg',             'fig_labels.jpg'),
    ('labels_correlogram.jpg', 'fig_labels_correlogram.jpg'),
    ('val_batch0_labels.jpg',  'fig_val_gt.jpg'),
    ('val_batch0_pred.jpg',    'fig_val_pred.jpg'),
]

print(f"Copying YOLOv5 figures into {REPORT_DIR}")
for src_name, dst_name in copy_map:
    src = Path(RUN_DIR) / src_name
    dst = REPORT_DIR / dst_name
    if src.exists():
        shutil.copyfile(src, dst)
        print(f"  {src_name:<28s} -> {dst_name}")
    else:
        print(f"  [skip] {src_name} not found in run directory")

# Final manifest of everything in report_figures/
print("\nContents of report_figures/:")
for f in sorted(REPORT_DIR.iterdir()):
    print(f"  {f.name:<32s} {f.stat().st_size//1024:>5d} KB")


In [ ]:
import pandas as pd
from pathlib import Path

RESULTS_CSV = Path(RUN_DIR) / 'results.csv'
results = pd.read_csv(RESULTS_CSV)
results.columns = [c.strip() for c in results.columns]

best_idx = results['metrics/mAP_0.5'].idxmax()
final = results.iloc[-1]
best  = results.iloc[best_idx]

rows = [
    ('Precision',       'metrics/precision'),
    ('Recall',          'metrics/recall'),
    ('mAP@0.5',         'metrics/mAP_0.5'),
    ('mAP@0.5:0.95',    'metrics/mAP_0.5:0.95'),
]

latex = []
latex.append(r'\begin{tabular}{lcc}')
latex.append(r'\toprule')
latex.append(r'Metric & Final Epoch & Best Epoch \\')
latex.append(r'\midrule')
for label, col in rows:
    latex.append(f'{label} & {final[col]:.4f} & {best[col]:.4f} \\\\')
latex.append(r'\bottomrule')
latex.append(r'\end{tabular}')

latex_str = '\n'.join(latex)
out_path = Path(RUN_DIR) / 'report_figures' / 'metrics_table.tex'
out_path.write_text(latex_str)

print(latex_str)
print(f"\nsaved -> {out_path}")
print(f"final epoch index: {len(results)-1}, best epoch index: {best_idx}")


In [ ]:
import shutil
from IPython.display import FileLink

archive_path = shutil.make_archive(
    base_name='/kaggle/working/report_figures',
    format='zip',
    root_dir=str(Path(RUN_DIR) / 'report_figures'),
)
print(f'Created archive: {archive_path}')
FileLink(archive_path)


In [ ]:
# Command for testing the model on an .mp4 video
import os

video_source = '/kaggle/input/datasets/divya26454/helmetlaptop/WIN_20260423_16_34_15_Pro.mp4'  # Update this path
output_project = '/kaggle/working/yolov5/runs/detect'
output_name = 'helmet_video'

!python detect.py --weights /kaggle/working/yolov5/runs/train/yolov5_rider_safety/weights/best.pt \
                  --img 640 --conf 0.4 --source "{video_source}" \
                  --project "{output_project}" --name "{output_name}" --exist-ok \
                  --classes 0 1 3

output_video_path = os.path.join(output_project, output_name, os.path.basename(video_source))
print('Processed video saved to:', output_video_path)

In [ ]:
# Display processed output video
from IPython.display import Video, display
import os

if os.path.exists(output_video_path):
    display(Video(output_video_path, embed=True))
else:
    print('No output video found at:', output_video_path)